# AlphaFold2 для строки 29

Откройте ноутбук в Google Colab, выберите GPU T4 и среду 2026.01 (Python 3.12), затем выполните все ячейки сверху вниз.
Установка и расчёт выполняются в отдельном окружении. Перезапуск ядра не требуется; backend Agg задан до первого импорта AlphaFold2.
Зафиксированы ревизия ColabFold и версия JAX из исходного успешного расчёта. Выполняются пять моделей AlphaFold2 PTM, seed 0, три рецикла, без шаблонов и релаксации.

Основа: https://github.com/sokrypton/ColabFold/blob/eb061aa6706959fdffe15e86d27ca08083d6ed8d/AlphaFold2.ipynb


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, math, hashlib, shutil, shlex
assert sys.version_info[:2] == (3, 12), 'Выберите среду Colab 2026.01 с Python 3.12'
assert Path('/content').is_dir(), 'Этот ноутбук предназначен для Google Colab'
subprocess.run(['nvidia-smi'], check=True)
sequence = 'MASVQQGEKQLFEKFWRGTFKAVATPRPESIIVASITARKPLPRTEPQNNPVVPAQDGPSEKLGQHLATEPLGTNSWERDKTCRELGATRGHSASHDKDL'
def run_live(command, **kwargs):
    print(command, flush=True)
    process = subprocess.Popen(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Command failed: ' + command)
work = Path('/content/row29-af2')
work.mkdir(exist_ok=True)
(work/'row29.fasta').write_text('>assignment5_row29_af2\n' + sequence + '\n')
run_live(f'{sys.executable} -m pip install virtualenv')
run_live(f'{sys.executable} -m virtualenv /content/row29-af2-env')
af_env = os.environ.copy()
af_env.pop('PYTHONPATH', None)
af_env['MPLBACKEND'] = 'Agg'
assert Path('/usr/lib64-nvidia').is_dir()
af_env['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'
run_live('/content/row29-af2-env/bin/python -m pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold@eb061aa6706959fdffe15e86d27ca08083d6ed8d" "jax[cuda12]==0.11.2"', env=af_env)


In [ ]:
probe = "import jax; import jax.numpy as j; import colabfold.batch; print(jax.__version__,jax.devices()); assert any(d.platform=='gpu' for d in jax.devices()); print(j.dot(j.ones((32,32)),j.ones((32,32))).block_until_ready()[0,0]); print('AlphaFold2 GPU and imports OK')"
run_live('/content/row29-af2-env/bin/python -c ' + shlex.quote(probe), cwd=work, env=af_env)
run_live('/content/row29-af2-env/bin/python -m pip freeze > dependencies.txt', cwd=work, env=af_env)


In [ ]:
run_live('/content/row29-af2-env/bin/colabfold_batch /content/row29-af2/row29.fasta /content/row29-af2/results --model-type alphafold2_ptm --num-models 5 --num-recycle 3 --num-relax 0 --msa-mode mmseqs2_uniref_env --random-seed 0', cwd=work, env=af_env)
models = sorted((work/'results').glob('*unrelaxed_rank_*.pdb'))
assert len(models) == 5, models
amino = dict(zip('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL'.split(), 'ARNDCQEGHILKMFPSTWYV'))
for path in models:
    atoms = [line for line in path.read_text().splitlines() if line.startswith('ATOM')]
    ca = [line for line in atoms if line[12:16].strip() == 'CA']
    assert len(ca) == 100 and {line[21] for line in ca} == {'A'}
    assert ''.join(amino[line[17:20]] for line in ca) == sequence
    assert all(math.isfinite(float(line[a:b])) for line in atoms for a,b in [(30,38),(38,46),(46,54)])
    print(path.name, 'residues', len(ca), 'mean_pLDDT', sum(float(line[60:66]) for line in ca)/100, 'sha256', hashlib.sha256(path.read_bytes()).hexdigest())
print('All five AlphaFold2 predictions validated')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'py3Dmol==2.5.5'], check=True)
import py3Dmol
best = sorted((work/'results').glob('*unrelaxed_rank_001_*.pdb'))[0]
view = py3Dmol.view(width=750, height=450)
view.addModel(best.read_text(), 'pdb')
view.setStyle({'cartoon': {'color': 'gray'}})
view.zoomTo()
view.show()


In [ ]:
from google.colab import files
archive_path = shutil.make_archive('/content/row29_alphafold2_results', 'zip', work)
files.download(archive_path)
